# Genie 4 in miniature: a toy world model with a guide agent

A world model predicts the next frame from the current state plus your action. This notebook simulates that loop in 2D: a latent state, action-conditioned transitions, drift, prompt-edit shocks, and a guide agent that checks each step and replans. All numbers are toy-scale — the real Genie 4 figures (2-minute coherence, native audio) are quoted in the post, not reproduced here.

In [ ]:
import numpy as np
rng = np.random.default_rng(7)

def run_world(steps=120, sigma=0.055, guided=False, edits=(30, 75)):
    # latent state: 8-dim 'scene description'; true dynamics are a slow rotation + action push
    state = np.zeros(8); true = np.zeros(8)
    coh = []
    for t in range(steps):
        action = np.array([np.sin(t / 20.0), np.cos(t / 25.0)] + [0] * 6) * 0.1
        true = 0.98 * true + action  # the world as it 'should' evolve
        state = 0.98 * state + action + rng.normal(0, sigma, 8)
        if t in edits:
            state += rng.normal(0, 0.35, 8)  # prompt edit shock: 'make it rain'
        if guided and np.linalg.norm(state - true) > 0.55:
            state = true + (state - true) * 0.35  # guide detects drift, replans
        coh.append(float(np.clip(1 - np.linalg.norm(state - true) / 3.0, 0, 1)))
    return np.array(coh)

ung = run_world(guided=False)
gui = run_world(guided=True)
print(f"unguided coherence at 120s: {ung[-1]:.2f}; guided: {gui[-1]:.2f}")
print(f"mean coherence unguided {ung.mean():.2f} vs guided {gui.mean():.2f}")
assert gui[-1] > ung[-1] and gui.mean() > ung.mean()

In [ ]:
# The headline arithmetic: Genie 3's ~60 s visual memory -> Genie 4's ~120 s reported coherence
genie3, genie4 = 60, 120
print(f"coherence window: {genie4 / genie3:.1f}x Genie 3")
assert genie4 / genie3 == 2.0
# Audio channels generated natively, per DeepMind: ambient, music, dialogue
channels = ["ambient", "music", "dialogue"]
print(f"native audio channels: {len(channels)} -> {channels}")
assert len(channels) == 3

In [ ]:
# Steering as state surgery: typing 'make it rain' edits the latent state mid-run
state = {"weather": "sunny", "characters": ["ana"], "camera": "eye-level"}
def steer(state, prompt):
    s = dict(state)
    if "rain" in prompt: s["weather"] = "rainy"
    if "add" in prompt and "dog" in prompt: s["characters"] = s["characters"] + ["dog"]
    return s
after = steer(steer(state, "make it rain"), "add a dog")
print(after)
assert after == {"weather": "rainy", "characters": ["ana", "dog"], "camera": "eye-level"}

In [ ]:
# Robot-training use case: the guide breaks a goal into steps and verifies each frame
goal = ["approach door", "open door", "enter room"]
frames_ok = [True, True, False]  # step 3 fails its frame check -> replan
plan = []
for step, ok in zip(goal, frames_ok):
    plan.append((step, "done" if ok else "REPLAN"))
    if not ok:
        plan.append((step + " (retry, new approach)", "done"))
print(plan)
assert plan[-1][1] == "done" and any("REPLAN" in p[1] for p in plan)